# Yelp Reviews Text Classification with BERT

Fine-tunes `bert-base-uncased` on the Yelp Reviews dataset to predict star ratings (1–5) from review text.

**Steps:**
1. Install & import dependencies  
2. Load and explore the data  
3. Subsample & split train / test  
4. Tokenize with BERT tokenizer  
5. Build PyTorch Dataset  
6. Load BERT classification model  
7. Configure & run training  
8. Evaluate — accuracy, classification report, confusion matrix  
9. Run inference on new text

## Step 1 — Install Dependencies

In [ ]:
%pip install transformers torch scikit-learn pyarrow -q

## Step 2 — Imports & Config

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import torch
from torch.utils.data import Dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# ── Configuration — adjust these as needed ────────────────────────────────────
DATA_PATH   = Path.cwd() / "Session 6" / "yelp_reviews.parquet"
MODEL_NAME  = "bert-base-uncased"
OUTPUT_DIR  = Path.cwd() / "Session 6" / "bert_yelp_output"

NUM_LABELS  = 5       # star ratings 1–5 encoded as 0–4
MAX_LEN     = 128     # max token length per review
SAMPLE_SIZE = 20_000  # rows to use (set to None for full 650k)
TEST_SIZE   = 0.2     # 80/20 split
BATCH_SIZE  = 16
NUM_EPOCHS  = 3
LR          = 2e-5
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Config OK")

## Step 3 — Load & Explore the Data

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f"Shape : {df.shape}")
print(f"Columns: {df.columns.tolist()}\n")
print("Label distribution (0 = ★1, 4 = ★5):")
print(df["label"].value_counts().sort_index())
df.head()

## Step 4 — Subsample & Train / Test Split

In [ ]:
# Stratified subsample so all 5 classes stay balanced
if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df["label"], random_state=SEED)
    print(f"Using {len(df):,} rows (stratified sample)")

X_train, X_test, y_train, y_test = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=TEST_SIZE,
    stratify=df["label"],
    random_state=SEED,
)

print(f"Train : {len(X_train):,}")
print(f"Test  : {len(X_test):,}")

## Step 5 — Tokenize with BERT Tokenizer

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

train_enc = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LEN)
test_enc  = tokenizer(X_test,  truncation=True, padding=True, max_length=MAX_LEN)

# Inspect a single tokenised example
sample_ids = train_enc["input_ids"][0]
print(f"Token IDs (first example): {sample_ids[:15]} …")
print(f"Decoded : {tokenizer.decode(sample_ids[:15])}")

## Step 6 — Build PyTorch Dataset

In [ ]:
class YelpDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = YelpDataset(train_enc, y_train)
test_dataset  = YelpDataset(test_enc,  y_test)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test  dataset: {len(test_dataset)} samples")
print(f"Sample keys  : {list(train_dataset[0].keys())}")

## Step 7 — Load BERT Classification Model

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## Step 8 — Configure & Run Training

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## Step 9 — Evaluate the Model

In [ ]:
results = trainer.evaluate()
print(f"Test accuracy: {results['eval_accuracy']:.4f}\n")

preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=[f"★{i+1}" for i in range(NUM_LABELS)]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"★{i+1}" for i in range(NUM_LABELS)],
            yticklabels=[f"★{i+1}" for i in range(NUM_LABELS)])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Yelp BERT Classifier")
plt.tight_layout()
plt.show()

## Step 10 — Save Model & Run Inference on New Text

In [ ]:
best_path = OUTPUT_DIR / "best_model"
trainer.save_model(str(best_path))
tokenizer.save_pretrained(str(best_path))
print(f"Model saved to {best_path}")

In [ ]:
def predict(texts, model, tokenizer, device="cpu"):
    """Return predicted star ratings (1–5) for a list of review strings."""
    model.eval()
    model.to(device)
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    predicted_labels = torch.argmax(logits, dim=1).cpu().numpy()
    return [label + 1 for label in predicted_labels]  # shift 0–4 → 1–5

reviews = [
    "Absolutely amazing food and fantastic service! Will definitely come back.",
    "Terrible experience. The food was cold and the waiter was rude.",
    "Pretty average place. Nothing special but not bad either.",
    "One of the best burgers I've ever had. Highly recommend!",
    "Waited 45 minutes for mediocre pasta. Very disappointing.",
]

stars = predict(reviews, model, tokenizer)
for review, star in zip(reviews, stars):
    print(f"★{star}  |  {review[:70]}")